In [74]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [75]:
df = pd.read_parquet("data/eda.parquet")
df = df.sort_index()

In [76]:
df["lag_1"] = df["Appliances"].shift(1)
df["lag_6"] = df["Appliances"].shift(6)
df["lag_144"] = df["Appliances"].shift(144)

In [77]:
df["rolling_mean_6"] = df["Appliances"].shift(1).rolling(6).mean()
df["rolling_std_6"] = df["Appliances"].shift(1).rolling(6).std()

df["rolling_mean_144"] = df["Appliances"].shift(1).rolling(144).mean()
df["rolling_std_144"] = df["Appliances"].shift(1).rolling(144).std()

In [78]:
df["hour"] = df.index.hour
df["day_of_week"] = df.index.dayofweek
df["is_weekend"] = (df.index.dayofweek >= 5).astype(int)

In [79]:
df["diff_1"] = df["Appliances"].shift(1).diff(1)
df["diff_6"] = df["Appliances"].shift(1).diff(6)
df["diff_144"] = df["Appliances"].shift(1).diff(144)

In [80]:
df.head(10)

,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,...,rolling_mean_6,rolling_std_6,rolling_mean_144,rolling_std_144,hour,day_of_week,is_weekend,diff_1,diff_6,diff_144
date,,,,,,,,,,,,,,,,,,,,,
2016-01-11 17:00:00,60,30,19.890000,47.596667,19.20,44.790000,19.79,44.730000,19.000000,45.566667,...,NaN,NaN,NaN,NaN,17,0,0,NaN,NaN,NaN
2016-01-11 17:10:00,60,30,19.890000,46.693333,19.20,44.722500,19.79,44.790000,19.000000,45.992500,...,NaN,NaN,NaN,NaN,17,0,0,NaN,NaN,NaN
2016-01-11 17:20:00,50,30,19.890000,46.300000,19.20,44.626667,19.79,44.933333,18.926667,45.890000,...,NaN,NaN,NaN,NaN,17,0,0,0.0,NaN,NaN
2016-01-11 17:30:00,50,40,19.890000,46.066667,19.20,44.590000,19.79,45.000000,18.890000,45.723333,...,NaN,NaN,NaN,NaN,17,0,0,-10.0,NaN,NaN
2016-01-11 17:40:00,60,40,19.890000,46.333333,19.20,44.530000,19.79,45.000000,18.890000,45.530000,...,NaN,NaN,NaN,NaN,17,0,0,0.0,NaN,NaN
2016-01-11 17:50:00,50,40,19.890000,46.026667,19.20,44.500000,19.79,44.933333,18.890000,45.730000,...,NaN,NaN,NaN,NaN,17,0,0,10.0,NaN,NaN
2016-01-11 18:00:00,60,50,19.890000,45.766667,19.20,44.500000,19.79,44.900000,18.890000,45.790000,...,55.000000,5.477226,NaN,NaN,18,0,0,-10.0,NaN,NaN
2016-01-11 18:10:00,60,50,19.856667,45.560000,19.20,44.500000,19.73,44.900000,18.890000,45.863333,...,55.000000,5.477226,NaN,NaN,18,0,0,10.0,0.0,NaN
2016-01-11 18:20:00,60,40,19.790000,45.597500,19.20,44.433333,19.73,44.790000,18.890000,45.790000,...,55.000000,5.477226,NaN,NaN,18,0,0,0.0,0.0,NaN


In [81]:
df.isna().sum()[df.isna().sum() > 0]

lag_1                 1
lag_6                 6
lag_144             144
rolling_mean_6        6
rolling_std_6         6
rolling_mean_144    144
rolling_std_144     144
diff_1                2
diff_6                7
diff_144            145
dtype: int64

In [82]:
df = df.dropna()

In [83]:
print(df.isna().sum().sum())
print(df.duplicated().sum())

0
0


In [84]:
from sklearn.model_selection import train_test_split

train_df, val_and_test_df = train_test_split(df, shuffle=False, test_size=0.3)
val_df, test_df  = train_test_split(val_and_test_df, shuffle=False, test_size=0.5)
print(f"{len(train_df) = }\n{len(val_df) = }\n{len(test_df) = }")

df = df.sort_index()

len(train_df) = 13713
len(val_df) = 2938
len(test_df) = 2939


In [85]:
X_train = train_df.drop(columns=["Appliances"])
X_val = val_df.drop(columns=["Appliances"])
X_test = test_df.drop(columns=["Appliances"])

y_train = train_df["Appliances"]
y_val = val_df["Appliances"]
y_test = test_df["Appliances"]

In [86]:
from sklearn.preprocessing import StandardScaler

scaler = RobustScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [87]:
import numpy as np

y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(y_train.to_numpy().reshape(-1, 1)).ravel()
y_val_scaled = y_scaler.transform(y_val.to_numpy().reshape(-1, 1)).ravel()
y_test_scaled = y_scaler.transform(y_test.to_numpy().reshape(-1, 1)).ravel()

In [88]:
def create_sequences(X, y, window=144):
    history = np.column_stack((X, y))
    X_seq = np.array([history[i-window:i] for i in range(window, len(y))])
    y_seq = y[window:]
    return X_seq, y_seq

In [89]:
X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled)
X_val_seq, y_val_seq = create_sequences(X_val_scaled, y_val_scaled)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_scaled)

In [90]:
Path("data").mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    "data/processed.npz",
    X_train=X_train_seq, y_train=y_train_seq,
    X_val=X_val_seq, y_val=y_val_seq,
    X_test=X_test_seq, y_test=y_test_seq
)

In [91]:
import joblib

joblib.dump(scaler, "data/feature_scaler.pkl")
joblib.dump(y_scaler, "data/target_scaler.pkl")

['data/target_scaler.pkl']